In [ ]:
print("Notebook 03 — Bayesian Optimization Controller")
print("Pipeline: 00a -> 01 -> 02 -> 03 (current)")
print("Cell 0 complete. Kernel is alive.")


In [ ]:
import numpy as np
import pandas as pd
import json
import subprocess
from pathlib import Path
import itertools
import warnings
import datetime

from scipy.stats import norm
from scipy.stats.qmc import LatinHypercube
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style='whitegrid', context='paper', font_scale=1.3)
sns.set_palette('viridis')


def ps_browse_file(title="Select file", filter_str="All files|*.*"):
    script = f"""
Add-Type -AssemblyName System.Windows.Forms
$d = New-Object System.Windows.Forms.OpenFileDialog
$d.Title = '{title}'
$d.Filter = '{filter_str}'
if ($d.ShowDialog() -eq 'OK') {{ $d.FileName }}
"""
    r = subprocess.run(["powershell.exe", "-NoProfile", "-Command", script],
                       capture_output=True, text=True)
    path = r.stdout.strip()
    if path:
        path = path.replace('\\', '/')
        if len(path) > 1 and path[1] == ':':
            drive = path[0].lower()
            path = f'/mnt/{drive}' + path[2:]
    return path or None


def ps_browse_folder(title="Select folder"):
    script = f"""
Add-Type -AssemblyName System.Windows.Forms
$d = New-Object System.Windows.Forms.FolderBrowserDialog
$d.Description = '{title}'
if ($d.ShowDialog() -eq 'OK') {{ $d.SelectedPath }}
"""
    r = subprocess.run(["powershell.exe", "-NoProfile", "-Command", script],
                       capture_output=True, text=True)
    path = r.stdout.strip()
    if path:
        path = path.replace('\\', '/')
        if len(path) > 1 and path[1] == ':':
            drive = path[0].lower()
            path = f'/mnt/{drive}' + path[2:]
    return path or None


print("Cell 1 complete — all imports loaded.")
%matplotlib inline


In [ ]:
# Safe defaults so downstream cells do not crash if run before Confirm is clicked
CAMPAIGN_NAME = 'baw_campaign_01'
MAX_TRIALS = 50
BATCH_SIZE = 5
N_TOP_DISPLAY = 10
ACQUISITION_FUNC = 'Expected Improvement'
XI = 0.01
USE_CONTACT_PRIOR = True
MIN_RESIDUES = 3
MAX_RESIDUES = 8
FORCE_INCLUDE_RESIDUES = [119, 312]
CONVERGENCE_PATIENCE = 10
CONVERGENCE_THRESHOLD = 0.1
NOTEBOOK02_PATH = Path.home() / 'BasementAntibodyWorks/notebooks/02_backbone_design.ipynb'
DESIGNS_DIR = Path.home() / 'BasementAntibodyWorks/designs'
HOTSPOT_JSON = Path.home() / 'BasementAntibodyWorks/data/ts/ts_hotspot_residues.json'

campaign_name_w = widgets.Text(value='baw_campaign_01', description='Campaign name:')
max_trials_w = widgets.IntSlider(value=50, min=10, max=200, description='Max trials:')
batch_size_w = widgets.Dropdown(options=[1, 3, 5, 10], value=5, description='Batch size:')
n_top_display_w = widgets.Dropdown(options=[5, 10, 20], value=10, description='Top N display:')
acquisition_w = widgets.Dropdown(
    options=['Expected Improvement', 'Upper Confidence Bound', 'Thompson Sampling'],
    value='Expected Improvement', description='Acquisition:')
xi_w = widgets.FloatSlider(value=0.01, min=0.0, max=0.5, step=0.01, description='xi (explore):')
use_contact_prior_w = widgets.Checkbox(value=True, description='Weight by contact scores')
min_residues_w = widgets.IntSlider(value=3, min=2, max=8, description='Min residues:')
max_residues_w = widgets.IntSlider(value=8, min=3, max=11, description='Max residues:')
force_include_w = widgets.Text(value='119,312', description='Force-include (PDB):')
convergence_patience_w = widgets.IntSlider(value=10, min=5, max=30, description='Patience:')
convergence_threshold_w = widgets.FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01,
    description='Min improvement:')

notebook02_path_display = widgets.Text(value='not set', description='Notebook 02:', disabled=True)
designs_dir_display = widgets.Text(value='not set', description='Designs dir:', disabled=True)
hotspot_json_display = widgets.Text(value='not set', description='Hotspot JSON:', disabled=True)

confirm_button = widgets.Button(description='Confirm', button_style='success')
output_area = widgets.Output()


def on_confirm_clicked(b):
    global CAMPAIGN_NAME, MAX_TRIALS, BATCH_SIZE, N_TOP_DISPLAY
    global ACQUISITION_FUNC, XI, USE_CONTACT_PRIOR
    global MIN_RESIDUES, MAX_RESIDUES, FORCE_INCLUDE_RESIDUES
    global CONVERGENCE_PATIENCE, CONVERGENCE_THRESHOLD
    global NOTEBOOK02_PATH, DESIGNS_DIR, HOTSPOT_JSON

    CAMPAIGN_NAME = campaign_name_w.value
    MAX_TRIALS = max_trials_w.value
    BATCH_SIZE = batch_size_w.value
    N_TOP_DISPLAY = n_top_display_w.value
    ACQUISITION_FUNC = acquisition_w.value
    XI = xi_w.value
    USE_CONTACT_PRIOR = use_contact_prior_w.value
    MIN_RESIDUES = min_residues_w.value
    MAX_RESIDUES = max_residues_w.value
    FORCE_INCLUDE_RESIDUES = [int(x.strip()) for x in force_include_w.value.split(',') if x.strip()]
    CONVERGENCE_PATIENCE = convergence_patience_w.value
    CONVERGENCE_THRESHOLD = convergence_threshold_w.value

    nb02 = ps_browse_file(title="Select Notebook 02 (backbone design)",
                           filter_str="Jupyter Notebook|*.ipynb")
    if nb02:
        NOTEBOOK02_PATH = Path(nb02)
        notebook02_path_display.value = str(NOTEBOOK02_PATH)

    designs = ps_browse_folder(title="Select designs directory")
    if designs:
        DESIGNS_DIR = Path(designs)
        designs_dir_display.value = str(DESIGNS_DIR)

    hotspot = ps_browse_file(title="Select hotspot residues JSON",
                              filter_str="JSON files|*.json")
    if hotspot:
        HOTSPOT_JSON = Path(hotspot)
        hotspot_json_display.value = str(HOTSPOT_JSON)

    with output_area:
        clear_output()
        print("CAMPAIGN CONFIGURATION CONFIRMED")
        print(f"Campaign name: {CAMPAIGN_NAME}")
        print(f"Max trials: {MAX_TRIALS}")
        print(f"Batch size: {BATCH_SIZE}")
        print(f"Top N display: {N_TOP_DISPLAY}")
        print(f"Acquisition function: {ACQUISITION_FUNC} (xi={XI})")
        print(f"Weight by contact scores: {USE_CONTACT_PRIOR}")
        print(f"Residue count range: [{MIN_RESIDUES}, {MAX_RESIDUES}]")
        print(f"Force-include residues: {FORCE_INCLUDE_RESIDUES}")
        print(f"Convergence patience: {CONVERGENCE_PATIENCE}")
        print(f"Convergence threshold: {CONVERGENCE_THRESHOLD}")
        print(f"Notebook 02 path: {NOTEBOOK02_PATH}")
        print(f"Designs dir: {DESIGNS_DIR}")
        print(f"Hotspot JSON: {HOTSPOT_JSON}")


confirm_button.on_click(on_confirm_clicked)

config_form = widgets.VBox([
    widgets.HTML("<h3>Campaign Configuration</h3>"),
    campaign_name_w, max_trials_w, batch_size_w, n_top_display_w,
    acquisition_w, xi_w, use_contact_prior_w,
    min_residues_w, max_residues_w, force_include_w,
    convergence_patience_w, convergence_threshold_w,
    notebook02_path_display, designs_dir_display, hotspot_json_display,
    confirm_button, output_area,
])
display(config_form)

print("Cell 2 complete — configuration widget ready. Click Confirm to set paths.")


In [ ]:
with open(HOTSPOT_JSON) as f:
    hotspot_data = json.load(f)

candidate_pool = hotspot_data['candidate_pool']

# Some hotspot JSONs (e.g. current ts_hotspot_residues.json) do not carry a
# contact_scores field yet -- fall back to uniform weighting rather than
# crashing, and say so clearly.
if 'contact_scores' in hotspot_data:
    contact_scores = {int(k): v for k, v in hotspot_data['contact_scores'].items()}
else:
    print("WARNING: 'contact_scores' not found in HOTSPOT_JSON — "
          "falling back to uniform contact scores (1.0) for all candidate residues.")
    contact_scores = {r: 1.0 for r in candidate_pool}

from itertools import combinations as itercombo

raw = []
for r in range(MIN_RESIDUES, MAX_RESIDUES + 1):
    for combo in itercombo(candidate_pool, r):
        if all(f in combo for f in FORCE_INCLUDE_RESIDUES):
            raw.append(tuple(sorted(combo)))
ALL_COMBINATIONS = raw

pool_index = {res: i for i, res in enumerate(candidate_pool)}
ALL_COMBINATIONS_ENCODED = np.zeros((len(ALL_COMBINATIONS), len(candidate_pool)), dtype=float)
for j, combo in enumerate(ALL_COMBINATIONS):
    for res in combo:
        ALL_COMBINATIONS_ENCODED[j, pool_index[res]] = 1.0

max_possible = sum(contact_scores.values())
COMBO_CONTACT_SCORES = np.array([
    sum(contact_scores.get(r, 0) for r in combo) / max_possible
    for combo in ALL_COMBINATIONS
])

print(f"Candidate pool ({len(candidate_pool)} residues): {candidate_pool}")
print(f"Contact scores: {contact_scores}")
print(f"Force-include residues: {FORCE_INCLUDE_RESIDUES}")
print(f"Combination constraints: min={MIN_RESIDUES}, max={MAX_RESIDUES} residues")
print(f"Valid combinations: {len(ALL_COMBINATIONS)} (out of {2 ** len(candidate_pool)} total)")

DESIGNS_DIR.mkdir(parents=True, exist_ok=True)

cs_sorted = sorted(contact_scores.items(), key=lambda kv: kv[1], reverse=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=[str(r) for r, _ in cs_sorted], y=[s for _, s in cs_sorted],
            hue=[str(r) for r, _ in cs_sorted], palette='viridis', legend=False, ax=ax)
ax.set_xlabel('Residue')
ax.set_ylabel('Contact score')
ax.set_title('Contact scores by residue')
fig.tight_layout()

fig_path = DESIGNS_DIR / 'contact_scores.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.savefig(fig_path.with_suffix('.svg'), format='svg', bbox_inches='tight')
plt.show()


In [ ]:
result_files = sorted(DESIGNS_DIR.rglob('trial_result.json'))

trial_records = []
for i, rf in enumerate(result_files):
    with open(rf) as f:
        data = json.load(f)
    combination = tuple(sorted(data['combination']))
    trial_records.append({
        'trial_id': i + 1,
        'combination': combination,
        'combination_str': ','.join(map(str, combination)),
        'best_pae': data['best_pae'],
        'n_passed': data.get('n_passed'),
        'n_total': data.get('n_total'),
        'output_dir': str(rf.parent),
    })

trials_df = pd.DataFrame(trial_records,
    columns=['trial_id', 'combination', 'combination_str', 'best_pae',
             'n_passed', 'n_total', 'output_dir'])

if len(trials_df) > 0:
    X_observed = np.array([
        [1.0 if r in combo else 0.0 for r in candidate_pool]
        for combo in trials_df['combination']
    ])
    y_observed = trials_df['best_pae'].to_numpy()
    print(f"Loaded {len(trials_df)} previous trial(s) from {DESIGNS_DIR}")
    print(f"Best pAE: {y_observed.min():.3f}")
    print(f"Worst pAE: {y_observed.max():.3f}")
    print(f"Mean pAE: {y_observed.mean():.3f}")
    display(trials_df.sort_values('best_pae')
            .style.background_gradient(subset=['best_pae'], cmap='viridis_r'))
else:
    X_observed = np.empty((0, len(candidate_pool)))
    y_observed = np.empty((0,))
    print("COLD START: no previous trial_result.json files found under "
          f"{DESIGNS_DIR}. Run Cell 5 to generate an initial Latin Hypercube batch.")


In [ ]:
if len(trials_df) >= BATCH_SIZE * 2:
    print("Sufficient data for GP. Run Cell 6.")
    NEXT_BATCH = []  # will be set by Cell 7
else:
    sampler = LatinHypercube(d=len(candidate_pool), seed=42)
    n_samples = max(BATCH_SIZE * 2, 20)
    raw_samples = sampler.random(n=n_samples)
    binary_samples = (raw_samples >= 0.5).astype(float)

    force_idx = [pool_index[r] for r in FORCE_INCLUDE_RESIDUES]
    for idx in force_idx:
        binary_samples[:, idx] = 1.0

    def nearest_combo(sample_vec):
        dists = np.sum((ALL_COMBINATIONS_ENCODED - sample_vec) ** 2, axis=1)
        return np.argmin(dists)

    candidate_indices = []
    for sample_vec in binary_samples:
        idx = nearest_combo(sample_vec)
        candidate_indices.append(idx)

    # Weight selection by contact score and de-duplicate, preserving diversity
    weighted_order = sorted(set(candidate_indices),
                             key=lambda i: COMBO_CONTACT_SCORES[i], reverse=True)

    n_select = BATCH_SIZE * 2
    if len(weighted_order) < n_select:
        remaining = [i for i in range(len(ALL_COMBINATIONS)) if i not in weighted_order]
        remaining_sorted = sorted(remaining, key=lambda i: COMBO_CONTACT_SCORES[i], reverse=True)
        weighted_order += remaining_sorted[: n_select - len(weighted_order)]

    selected_indices = weighted_order[:n_select]
    NEXT_BATCH = [ALL_COMBINATIONS[i] for i in selected_indices]

    lhs_df = pd.DataFrame({
        'combination': NEXT_BATCH,
        'combination_str': [','.join(map(str, c)) for c in NEXT_BATCH],
        'contact_score_sum': [COMBO_CONTACT_SCORES[i] for i in selected_indices],
        'n_residues': [len(c) for c in NEXT_BATCH],
    })

    print(f"COLD START: selected {len(NEXT_BATCH)} diverse combinations via Latin Hypercube Sampling")
    display(lhs_df.sort_values('contact_score_sum', ascending=False))


In [ ]:
if len(y_observed) < 3:
    print("ERROR: Need at least 3 trial results. Run more trials first.")
    raise RuntimeError(
        f"Cannot fit GP surrogate: only {len(y_observed)} trial result(s) available, "
        "need at least 3. Complete more trials with Notebook 02 and re-run this notebook."
    )

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True, random_state=42)
gp.fit(X_observed, y_observed)

y_pred, y_std = gp.predict(ALL_COMBINATIONS_ENCODED, return_std=True)

tested_set = set(trials_df['combination_str']) if len(trials_df) else set()
combo_strs = [','.join(map(str, c)) for c in ALL_COMBINATIONS]
already_tested = np.array([s in tested_set for s in combo_strs])

gp_df = pd.DataFrame({
    'combination': ALL_COMBINATIONS,
    'combination_str': combo_strs,
    'predicted_pae': y_pred,
    'uncertainty': y_std,
    'already_tested': already_tested,
    'contact_score_sum': COMBO_CONTACT_SCORES,
    'n_residues': [len(c) for c in ALL_COMBINATIONS],
})

train_r2 = gp.score(X_observed, y_observed)
print("GP FIT SUMMARY")
print(f"Kernel: {gp.kernel_}")
print(f"Training points: {len(X_observed)}")
print(f"Training R^2: {train_r2:.3f}")
print(f"\nTop {N_TOP_DISPLAY} predicted combinations (lowest predicted pAE):")
display(gp_df.sort_values('predicted_pae').head(N_TOP_DISPLAY)
        [['combination_str', 'predicted_pae', 'uncertainty', 'contact_score_sum']])

untested_plot = gp_df[~gp_df['already_tested']]
tested_plot = gp_df[gp_df['already_tested']]

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=untested_plot, x='predicted_pae', y='uncertainty',
                 hue='contact_score_sum', palette='viridis', ax=ax)
if len(tested_plot) > 0:
    ax.scatter(tested_plot['predicted_pae'], tested_plot['uncertainty'],
               marker='x', s=80, color='red', label='Tested', zorder=5)
ax.set_xlabel('Predicted pAE')
ax.set_ylabel('Uncertainty (std)')
ax.set_title('GP landscape')
ax.legend(fontsize=8)
fig.tight_layout()

gp_fig_path = DESIGNS_DIR / 'gp_landscape.png'
plt.savefig(gp_fig_path, dpi=300, bbox_inches='tight')
plt.savefig(gp_fig_path.with_suffix('.svg'), format='svg', bbox_inches='tight')
plt.show()

print(f"Cell 6 complete. GP fitted on {len(X_observed)} observations. Run Cell 7 for acquisition.")


In [ ]:
from scipy.stats import norm

untested_mask = ~gp_df['already_tested'].values
y_pred_untested = y_pred[untested_mask]
y_std_untested = y_std[untested_mask]
y_best = float(np.min(y_observed))

if ACQUISITION_FUNC == 'Expected Improvement':
    z = (y_best - y_pred_untested - XI) / (y_std_untested + 1e-9)
    acq = (y_best - y_pred_untested - XI) * norm.cdf(z) + y_std_untested * norm.pdf(z)
    acq = np.maximum(acq, 0)

elif ACQUISITION_FUNC == 'Upper Confidence Bound':
    kappa = 2.0
    acq = -(y_pred_untested - kappa * y_std_untested)

elif ACQUISITION_FUNC == 'Thompson Sampling':
    np.random.seed(42)
    y_sample = np.random.normal(y_pred_untested, y_std_untested)
    acq = -y_sample

if USE_CONTACT_PRIOR:
    contact_weights = COMBO_CONTACT_SCORES[untested_mask]
    acq = acq * (0.5 + 0.5 * contact_weights)

untested_gp_df = gp_df[untested_mask].reset_index(drop=True)
acq_df = pd.DataFrame({
    'combination': untested_gp_df['combination'],
    'combination_str': untested_gp_df['combination_str'],
    'predicted_pae': y_pred_untested,
    'uncertainty': y_std_untested,
    'acquisition_score': acq,
    'contact_score_sum': untested_gp_df['contact_score_sum'],
    'n_residues': untested_gp_df['n_residues'],
})
acq_df = acq_df.sort_values('acquisition_score', ascending=False).reset_index(drop=True)

NEXT_BATCH = [tuple(c) for c in acq_df.head(BATCH_SIZE)['combination']]

print(f"Acquisition function: {ACQUISITION_FUNC}, xi={XI}, "
      f"contact prior={'on' if USE_CONTACT_PRIOR else 'off'}")
print(f"Current best pAE: {y_best:.3f}")
print(f"Next batch selected: {len(NEXT_BATCH)} combinations")
display(acq_df.head(BATCH_SIZE)
        [['combination_str', 'predicted_pae', 'uncertainty', 'acquisition_score']])

top20 = acq_df.head(20)
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=top20, x='acquisition_score', y='combination_str',
            hue='predicted_pae', dodge=False, palette='coolwarm', ax=ax)
ax.set_title('Top 20 Acquisition Scores')
ax.set_xlabel('Acquisition score')
ax.set_ylabel('Combination')
fig.tight_layout()

acq_fig_path = DESIGNS_DIR / 'acquisition_scores.png'
plt.savefig(acq_fig_path, dpi=300, bbox_inches='tight')
plt.savefig(acq_fig_path.with_suffix('.svg'), format='svg', bbox_inches='tight')
plt.show()

print(f"Cell 7 complete. NEXT_BATCH set with {len(NEXT_BATCH)} combinations. Run Cell 8 for dashboard.")


In [ ]:
if len(trials_df) == 0:
    print("No trials yet. Run Cell 9 to execute the first batch, then return here.")
else:
    results_sorted = trials_df.sort_values('trial_id').copy()
    results_sorted['running_best'] = results_sorted['best_pae'].cummin()
    top_quartile = trials_df[trials_df['best_pae'] <= trials_df['best_pae'].quantile(0.25)]
    n_residues_col = trials_df['combination'].apply(
        lambda c: len(c) if isinstance(c, (list, tuple)) else len(str(c).split(',')))

    fig, ax = plt.subplots(2, 3, figsize=(18, 12))

    # Plot 1: learning curve
    sns.lineplot(data=results_sorted, x='trial_id', y='running_best', ax=ax[0, 0],
                 color='steelblue', linewidth=2)
    sns.scatterplot(data=results_sorted, x='trial_id', y='best_pae', ax=ax[0, 0],
                     alpha=0.6, color='coral', s=60)
    ax[0, 0].axhline(y=10.0, color='red', linestyle='--', alpha=0.5, label='threshold 10.0')
    ax[0, 0].set_title('Learning Curve')
    ax[0, 0].set_xlabel('Trial number')
    ax[0, 0].set_ylabel('Best pAE')
    ax[0, 0].legend(fontsize=8)

    # Plot 2: score distribution
    sns.histplot(data=trials_df, x='best_pae', bins=15, kde=True, ax=ax[0, 1], color='steelblue')
    ax[0, 1].axvline(x=trials_df['best_pae'].min(), color='green', linestyle='--', label='current best')
    ax[0, 1].axvline(x=10.0, color='red', linestyle='--', label='threshold')
    ax[0, 1].set_title('Score Distribution')
    ax[0, 1].set_xlabel('Best pAE')
    ax[0, 1].legend(fontsize=8)

    # Plot 3: residue importance
    importance = {}
    for res in candidate_pool:
        count = sum(1 for combo in top_quartile['combination'] if res in combo)
        importance[res] = count
    imp_df = pd.DataFrame({'residue': list(importance.keys()),
                            'count': list(importance.values()),
                            'contact_score': [contact_scores.get(r, 0) for r in importance.keys()]})
    imp_df = imp_df.sort_values('count', ascending=False)
    sns.barplot(data=imp_df, x='residue', y='count', hue='contact_score', ax=ax[0, 2],
                palette='viridis', dodge=False)
    ax[0, 2].set_title('Residue Importance (top 25% trials)')
    ax[0, 2].set_xlabel('Residue (PDB number)')
    ax[0, 2].set_ylabel('Frequency in top trials')
    leg = ax[0, 2].get_legend()
    if leg is not None:
        leg.remove()

    # Plot 4: pairwise co-occurrence heatmap
    pool_list = list(candidate_pool)
    co_matrix = np.zeros((len(pool_list), len(pool_list)))
    for combo in top_quartile['combination']:
        combo_list = list(combo) if isinstance(combo, (list, tuple)) else \
            [int(x) for x in str(combo).strip('[]').split(',')]
        for i, r1 in enumerate(pool_list):
            for j, r2 in enumerate(pool_list):
                if r1 in combo_list and r2 in combo_list:
                    co_matrix[i, j] += 1
    co_df = pd.DataFrame(co_matrix, index=pool_list, columns=pool_list)
    sns.heatmap(co_df, ax=ax[1, 0], cmap='Blues', annot=True, fmt='.0f', linewidths=0.5)
    ax[1, 0].set_title('Residue Co-occurrence (top 25%)')

    # Plot 5: score vs combination size
    trials_df['n_residues'] = n_residues_col
    sns.boxplot(data=trials_df, x='n_residues', y='best_pae', ax=ax[1, 1], hue='n_residues',
                palette='coolwarm', legend=False)
    sns.stripplot(data=trials_df, x='n_residues', y='best_pae', ax=ax[1, 1],
                  color='black', alpha=0.5, size=4)
    ax[1, 1].set_title('Score vs Combination Size')
    ax[1, 1].set_xlabel('Number of residues')
    ax[1, 1].set_ylabel('Best pAE')

    # Plot 6: GP predicted vs actual
    if len(y_observed) > 0:
        y_gp_pred_train, _ = gp.predict(X_observed, return_std=True)
        pred_vs_actual = pd.DataFrame({'actual': y_observed, 'predicted': y_gp_pred_train})
        sns.scatterplot(data=pred_vs_actual, x='actual', y='predicted', ax=ax[1, 2],
                         s=80, color='steelblue')
        min_val = min(y_observed.min(), y_gp_pred_train.min()) - 0.5
        max_val = max(y_observed.max(), y_gp_pred_train.max()) + 0.5
        ax[1, 2].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7, label='perfect')
        ax[1, 2].set_title('GP: Predicted vs Actual')
        ax[1, 2].set_xlabel('Actual pAE')
        ax[1, 2].set_ylabel('GP predicted pAE')
        ax[1, 2].legend()

    plt.suptitle(f'Campaign Dashboard — {CAMPAIGN_NAME}', fontsize=14, fontweight='bold')
    plt.tight_layout()

    dashboard_path = DESIGNS_DIR / 'campaign_dashboard.png'
    plt.savefig(dashboard_path, dpi=300, bbox_inches='tight')
    plt.savefig(dashboard_path.with_suffix('.svg'), format='svg', bbox_inches='tight')
    plt.show()
    plt.close()

    best_idx = trials_df['best_pae'].idxmin()
    print(f"Dashboard saved: {dashboard_path}")
    print(f"Current best: {trials_df.loc[best_idx, 'best_pae']:.3f} "
          f"(trial {trials_df.loc[best_idx, 'trial_id']}, "
          f"combination {list(trials_df.loc[best_idx, 'combination'])})")
    print(f"Trials completed: {len(trials_df)} / {MAX_TRIALS}")
    print("Run Cell 9 to execute next batch.")

# Also update uncertainty map if GP has been fitted
try:
    if 'update_uncertainty_map' in dir() and len(y_observed) > 0:
        update_uncertainty_map(map_output_widget=map_live_output if 'map_live_output' in dir() else None)
except Exception as _ump_err:
    print(f'Uncertainty map skipped (run Cell 8b first): {_ump_err}')


In [ ]:
# Cell 8b: Live uncertainty map — 2D PCA projection of the combinatorial space
#
# Projects all possible residue combinations onto 2 principal components
# using the binary encoding matrix. Each combination becomes a point in 2D.
# Colour = GP uncertainty (sigma). Red = unexplored. Blue = well characterised.
#
# The map updates live after every trial in Cell 9 by calling
# update_uncertainty_map() from the run callback.
#
# PCA projection is computed once here and reused for all updates.
# update_uncertainty_map() is defined here and called from Cell 9.

from sklearn.decomposition import PCA

# ── Compute PCA projection once ────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(ALL_COMBINATIONS_ENCODED)

combo_strs_all = [','.join(map(str, c)) for c in ALL_COMBINATIONS]

print(f'PCA projection computed.')
print(f'  PC1 variance explained : {pca.explained_variance_ratio_[0]:.1%}')
print(f'  PC2 variance explained : {pca.explained_variance_ratio_[1]:.1%}')
print(f'  Total variance captured: {pca.explained_variance_ratio_.sum():.1%}')
print(f'  Combinations projected : {len(pca_coords)}')
print()
print('Uncertainty map will update live after each trial in Cell 9.')
print('Colour scale: RED = unexplored/uncertain   BLUE = well characterised')
print()

# ── Define live update function ─────────────────────────────────────────────────
def update_uncertainty_map(map_output_widget=None):
    """
    Recompute GP predictions with latest data and redraw the uncertainty map.
    Called after every trial completes in Cell 9.

    If map_output_widget is provided, renders inside that Output widget.
    Otherwise renders inline.
    """
    # Recompute GP predictions with current X_observed, y_observed
    if len(y_observed) == 0:
        print('No trials yet — uncertainty map will appear after first trial.')
        return

    y_pred_map, y_std_map = gp.predict(ALL_COMBINATIONS_ENCODED, return_std=True)

    # Build tested set
    tested_set_map = set(trials_df['combination_str']) if len(trials_df) > 0 else set()
    tested_mask_map = np.array([s in tested_set_map for s in combo_strs_all])

    # Build next batch set
    next_batch_strs = set(','.join(map(str, sorted(c))) for c in NEXT_BATCH) if NEXT_BATCH else set()

    fig, ax = plt.subplots(figsize=(12, 8))

    # ── All combinations coloured by GP uncertainty ───────────────────────────
    # RdYlBu_r: red=high uncertainty, yellow=medium, blue=low uncertainty
    sc = ax.scatter(
        pca_coords[:, 0], pca_coords[:, 1],
        c=y_std_map,
        cmap='RdYlBu_r',
        s=18,
        alpha=0.7,
        vmin=0,
        vmax=max(y_std_map.max(), 0.01),
        zorder=2,
    )
    cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
    cbar.set_label('GP Uncertainty (σ)', fontsize=10)

    # ── Tested combinations — white X ────────────────────────────────────────
    if tested_mask_map.any():
        ax.scatter(
            pca_coords[tested_mask_map, 0],
            pca_coords[tested_mask_map, 1],
            marker='x', s=100, c='white', linewidths=2.0,
            zorder=5, label=f'Tested ({tested_mask_map.sum()})',
        )

    # ── Current best — gold star ──────────────────────────────────────────────
    if len(y_observed) > 0:
        best_idx_obs = int(np.argmin(y_observed))
        best_combo = trials_df.iloc[best_idx_obs]['combination_str']
        best_mask_map = np.array([s == best_combo for s in combo_strs_all])
        if best_mask_map.any():
            ax.scatter(
                pca_coords[best_mask_map, 0],
                pca_coords[best_mask_map, 1],
                marker='*', s=500, c='gold', edgecolors='black', linewidths=1.0,
                zorder=10,
                label=f'Best pAE={y_observed.min():.2f}',
            )

    # ── Next batch — green open circles ──────────────────────────────────────
    if next_batch_strs:
        nb_mask_map = np.array([s in next_batch_strs for s in combo_strs_all])
        if nb_mask_map.any():
            ax.scatter(
                pca_coords[nb_mask_map, 0],
                pca_coords[nb_mask_map, 1],
                marker='o', s=220, facecolors='none',
                edgecolors='lime', linewidths=2.5,
                zorder=8,
                label=f'Next batch ({nb_mask_map.sum()})',
            )

    # ── Labels and title ──────────────────────────────────────────────────────
    n_trials = len(y_observed)
    best_pae = y_observed.min() if n_trials > 0 else float('nan')
    pct_covered = 100 * tested_mask_map.sum() / len(ALL_COMBINATIONS)

    ax.set_title(
        f'Combinatorial Space — Uncertainty Map\n'
        f'Trial {n_trials} / {MAX_TRIALS}  |  '
        f'Best pAE: {best_pae:.3f}  |  '
        f'Space covered: {pct_covered:.1f}%  |  '
        f'RED = unexplored   BLUE = well characterised',
        fontsize=10,
    )
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=9)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=9)
    ax.legend(loc='upper right', fontsize=8, framealpha=0.8)

    plt.tight_layout()

    # Save PNG and SVG with trial number in filename
    map_path = DESIGNS_DIR / f'uncertainty_map_trial_{n_trials:03d}.png'
    plt.savefig(map_path, dpi=150, bbox_inches='tight')
    plt.savefig(map_path.with_suffix('.svg'), format='svg', bbox_inches='tight')

    if map_output_widget is not None:
        with map_output_widget:
            clear_output(wait=True)
            plt.show()
    else:
        plt.show()

    plt.close()
    print(f'Uncertainty map saved: {map_path.name}')


# ── Output widget for live updates ───────────────────────────────────────────
map_live_output = widgets.Output()
display(map_live_output)

# ── Draw initial map ──────────────────────────────────────────────────────────
if len(y_observed) > 0:
    update_uncertainty_map(map_output_widget=map_live_output)
else:
    with map_live_output:
        print('No trials yet. Map will appear after first trial completes in Cell 9.')
        print('Re-run this cell after running Cell 9 to see the initial map.')

print('Cell 8b complete. update_uncertainty_map() is ready for Cell 9.')


In [ ]:
run_btn = widgets.Button(description='Run batch', button_style='success')
skip_btn = widgets.Button(description='Mark as manually run', button_style='warning')
run_out = widgets.Output()


def _lookup_acq(combination, col):
    match = acq_df[acq_df['combination'].apply(lambda c: tuple(c) == tuple(combination))]
    if len(match) > 0:
        return match[col].values[0]
    return None


def on_run_clicked(b):
    try:
        import papermill as pm
        HAS_PAPERMILL = True
    except ImportError:
        HAS_PAPERMILL = False

    for combination in NEXT_BATCH:
        trial_id = len(list(DESIGNS_DIR.rglob('trial_result.json'))) + 1
        out_dir = DESIGNS_DIR / f'trial_{trial_id:04d}'
        out_dir.mkdir(parents=True, exist_ok=True)

        with run_out:
            print(f'Running trial {trial_id}: combination {list(combination)}...')

        if HAS_PAPERMILL:
            try:
                pm.execute_notebook(
                    str(NOTEBOOK02_PATH),
                    str(out_dir / 'executed_nb02.ipynb'),
                    parameters={
                        'TRIAL_COMBINATION': list(combination),
                        'CANDIDATE_POOL_JSON': str(HOTSPOT_JSON),
                    },
                    cwd=str(NOTEBOOK02_PATH.parent)
                )
            except Exception as e:
                with run_out:
                    print(f'ERROR in trial {trial_id}: {e}')
                continue
        else:
            with run_out:
                print(f'papermill not available. Install with: pip install papermill')
                print(f'Alternatively run Notebook 02 manually with:')
                print(f'  TRIAL_COMBINATION = {list(combination)}')
                print(f'  CANDIDATE_POOL_JSON = "{HOTSPOT_JSON}"')
            continue

        result_path = out_dir / 'trial_result.json'
        if result_path.exists():
            with open(result_path) as f:
                result = json.load(f)
            with run_out:
                print(f'Trial {trial_id} complete. Score: {result.get("best_pae", "?"):.2f}')

            with run_out:
                print(f'  Updating GP and uncertainty map...')
                # Reload all trials and refit GP for live map update
                _all_results = sorted(DESIGNS_DIR.rglob('trial_result.json'))
                _trial_combos = []
                _trial_scores = []
                for _rf in _all_results:
                    with open(_rf) as _f:
                        _d = json.load(_f)
                    _trial_combos.append(tuple(sorted(_d['combination'])))
                    _trial_scores.append(_d['best_pae'])
                if len(_trial_scores) >= 1:
                    global X_observed, y_observed, trials_df
                    X_observed = np.array([
                        [1.0 if r in c else 0.0 for r in candidate_pool]
                        for c in _trial_combos
                    ])
                    y_observed = np.array(_trial_scores)
                    trials_df = pd.DataFrame({
                        'combination': _trial_combos,
                        'combination_str': [','.join(map(str, c)) for c in _trial_combos],
                        'best_pae': _trial_scores,
                    })
                    trials_df_live = trials_df
                    try:
                        gp.fit(X_observed, y_observed)
                        update_uncertainty_map(map_output_widget=map_live_output)
                    except Exception as _map_err:
                        print(f'  Map update skipped: {_map_err}')

            checkpoint = {
                'campaign_name': CAMPAIGN_NAME,
                'trials_completed': trial_id,
                'timestamp': datetime.datetime.now().isoformat(),
            }
            with open(DESIGNS_DIR / 'campaign_state.json', 'w') as f:
                json.dump(checkpoint, f, indent=2)
        else:
            with run_out:
                print(f'WARNING: trial_result.json not found for trial {trial_id}')

    with run_out:
        print('Final uncertainty map update...')
        try:
            update_uncertainty_map(map_output_widget=map_live_output)
        except Exception as _e:
            print(f'Final map update skipped: {_e}')

    with run_out:
        print()
        print('Batch complete. Run Cell 10 to check convergence and update campaign state.')


def on_skip_clicked(b):
    with run_out:
        print('Skipped execution. Run Cell 10 to reload results from disk.')


run_btn.on_click(on_run_clicked)
skip_btn.on_click(on_skip_clicked)

batch_table = pd.DataFrame([
    {'combination': str(list(c)),
     'predicted_pae': f'{_lookup_acq(c, "predicted_pae"):.2f}' if _lookup_acq(c, "predicted_pae") is not None else "?",
     'acquisition_score': f'{_lookup_acq(c, "acquisition_score"):.4f}' if _lookup_acq(c, "acquisition_score") is not None else "?"}
    for c in NEXT_BATCH
])

display(widgets.VBox([
    widgets.HTML(f'<h3>Next Batch — {len(NEXT_BATCH)} combinations</h3>'),
    widgets.HTML(f'<i>Estimated runtime: {len(NEXT_BATCH) * 40 * 1.5:.0f} minutes on RTX 4070</i>'),
    widgets.HTML('<b>Combinations to test:</b>'),
    widgets.HTML(batch_table.to_html(index=False)),
    widgets.HBox([run_btn, skip_btn]),
    run_out,
]))


In [ ]:
result_files = sorted(DESIGNS_DIR.rglob('trial_result.json'))
trial_records = []
for i, rf in enumerate(result_files):
    with open(rf) as f:
        data = json.load(f)
    combination = tuple(sorted(data['combination']))
    trial_records.append({
        'trial_id': i + 1,
        'combination': combination,
        'combination_str': ','.join(map(str, combination)),
        'best_pae': data['best_pae'],
        'n_passed': data.get('n_passed'),
        'n_total': data.get('n_total'),
        'output_dir': str(rf.parent),
    })

trials_df = pd.DataFrame(trial_records,
    columns=['trial_id', 'combination', 'combination_str', 'best_pae',
             'n_passed', 'n_total', 'output_dir'])

if len(trials_df) > 0:
    X_observed = np.array([
        [1.0 if r in combo else 0.0 for r in candidate_pool]
        for combo in trials_df['combination']
    ])
    y_observed = trials_df['best_pae'].to_numpy()
else:
    X_observed = np.empty((0, len(candidate_pool)))
    y_observed = np.empty((0,))

CONVERGED = False
BUDGET_EXHAUSTED = len(y_observed) >= MAX_TRIALS

if len(y_observed) >= CONVERGENCE_PATIENCE:
    recent_scores = y_observed[-CONVERGENCE_PATIENCE:]
    overall_best = float(np.min(y_observed))
    recent_best = float(np.min(recent_scores))
    improvement = recent_best - overall_best  # negative = got worse, 0 = no improvement
    if abs(improvement) < CONVERGENCE_THRESHOLD:
        CONVERGED = True

campaign_state = {
    'campaign_name': CAMPAIGN_NAME,
    'trials_completed': len(trials_df),
    'best_pae': float(np.min(y_observed)) if len(y_observed) > 0 else None,
    'best_combination': list(trials_df.loc[trials_df['best_pae'].idxmin(), 'combination']) if len(trials_df) > 0 else [],
    'converged': CONVERGED,
    'budget_exhausted': BUDGET_EXHAUSTED,
    'timestamp': datetime.datetime.now().isoformat(),
}
with open(DESIGNS_DIR / 'campaign_state.json', 'w') as f:
    json.dump(campaign_state, f, indent=2)

print("CONVERGENCE CHECK")
print(f"Trials completed: {len(trials_df)} / {MAX_TRIALS}")
if len(trials_df) > 0:
    best_idx = trials_df['best_pae'].idxmin()
    print(f"Best pAE: {trials_df.loc[best_idx, 'best_pae']:.2f} (trial {trials_df.loc[best_idx, 'trial_id']})")
else:
    print("Best pAE: n/a (no trials yet)")
print(f"Converged: {'YES' if CONVERGED else 'NO'}")
print(f"Budget exhausted: {'YES' if BUDGET_EXHAUSTED else 'NO'}")
print()
if not CONVERGED and not BUDGET_EXHAUSTED:
    print('Run Cell 6 to fit updated GP and select next batch.')
else:
    print('Run Cell 11 for final campaign summary.')


In [ ]:
with open(DESIGNS_DIR / 'campaign_state.json') as f:
    campaign_state = json.load(f)

result_files = sorted(DESIGNS_DIR.rglob('trial_result.json'))
trial_records = []
for i, rf in enumerate(result_files):
    with open(rf) as f:
        data = json.load(f)
    combination = tuple(sorted(data['combination']))
    trial_records.append({
        'trial_id': i + 1,
        'combination': combination,
        'combination_str': ','.join(map(str, combination)),
        'best_pae': data['best_pae'],
        'n_passed': data.get('n_passed'),
        'n_total': data.get('n_total'),
        'output_dir': str(rf.parent),
    })
trials_df = pd.DataFrame(trial_records,
    columns=['trial_id', 'combination', 'combination_str', 'best_pae',
             'n_passed', 'n_total', 'output_dir'])

print("FULL RANKED TRIAL TABLE")
display(trials_df.sort_values('best_pae').reset_index(drop=True))

print("\nTOP 10 COMBINATIONS")
display(trials_df.nsmallest(10, 'best_pae')
        [['trial_id', 'combination_str', 'best_pae']].reset_index(drop=True))

top_quartile = trials_df[trials_df['best_pae'] <= trials_df['best_pae'].quantile(0.25)]
importance = {}
for res in candidate_pool:
    count = sum(1 for combo in top_quartile['combination'] if res in combo)
    importance[res] = count
imp_df = pd.DataFrame({'residue': list(importance.keys()), 'count': list(importance.values())})
imp_df = imp_df.sort_values('count', ascending=False)
print("\nRESIDUE IMPORTANCE (frequency in top 25% of trials)")
display(imp_df.reset_index(drop=True))

CONVERGED = campaign_state.get('converged', False)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: complete learning curve, convergence region shaded if CONVERGED
results_sorted = trials_df.sort_values('trial_id').copy()
results_sorted['running_best'] = results_sorted['best_pae'].cummin()
sns.lineplot(data=results_sorted, x='trial_id', y='running_best', ax=axes[0, 0],
             color='steelblue', linewidth=2)
sns.scatterplot(data=results_sorted, x='trial_id', y='best_pae', ax=axes[0, 0],
                 alpha=0.6, color='coral', s=60)
if CONVERGED and len(results_sorted) >= CONVERGENCE_PATIENCE:
    conv_start = results_sorted['trial_id'].iloc[-CONVERGENCE_PATIENCE]
    conv_end = results_sorted['trial_id'].iloc[-1]
    axes[0, 0].axvspan(conv_start, conv_end, color='green', alpha=0.15, label='converged region')
    axes[0, 0].legend(fontsize=8)
axes[0, 0].set_title('Complete Learning Curve')
axes[0, 0].set_xlabel('Trial number')
axes[0, 0].set_ylabel('Best pAE')

# Plot 2: final residue importance barplot
sns.barplot(data=imp_df, x='residue', y='count', hue='residue', palette='viridis',
            legend=False, ax=axes[0, 1])
axes[0, 1].set_title('Final Residue Importance (top 25% trials)')
axes[0, 1].set_xlabel('Residue (PDB number)')
axes[0, 1].set_ylabel('Frequency in top trials')

# Plot 3: pairwise co-occurrence for all top-quartile trials
pool_list = list(candidate_pool)
co_matrix = np.zeros((len(pool_list), len(pool_list)))
for combo in top_quartile['combination']:
    combo_list = list(combo)
    for i, r1 in enumerate(pool_list):
        for j, r2 in enumerate(pool_list):
            if r1 in combo_list and r2 in combo_list:
                co_matrix[i, j] += 1
co_df = pd.DataFrame(co_matrix, index=pool_list, columns=pool_list)
sns.heatmap(co_df, ax=axes[1, 0], cmap='Blues', annot=True, fmt='.0f', linewidths=0.5)
axes[1, 0].set_title('Residue Co-occurrence (top 25%, final)')

# Plot 4: score distribution comparison — first half vs second half (violin)
half = len(trials_df) // 2
trials_df_half = trials_df.copy()
trials_df_half['half'] = ['first' if i < half else 'second' for i in range(len(trials_df_half))]
sns.violinplot(data=trials_df_half, x='half', y='best_pae', hue='half', palette='coolwarm',
               legend=False, ax=axes[1, 1])
axes[1, 1].set_title('Score Distribution: First Half vs Second Half')
axes[1, 1].set_xlabel('Campaign progress')
axes[1, 1].set_ylabel('Best pAE')

plt.tight_layout()

final_path = DESIGNS_DIR / 'campaign_final_report.png'
plt.savefig(final_path, dpi=300, bbox_inches='tight')
plt.savefig(final_path.with_suffix('.svg'), format='svg', bbox_inches='tight')
plt.show()

top5 = trials_df.nsmallest(5, 'best_pae')
export = [{'rank': i + 1, 'combination': list(row['combination']),
           'best_pae': row['best_pae'], 'output_dir': row.get('output_dir', '')}
          for i, (_, row) in enumerate(top5.iterrows())]
with open(DESIGNS_DIR / 'top_combinations_for_validation.json', 'w') as f:
    json.dump(export, f, indent=2)

print("Campaign summary complete.")
print("Top 5 combinations exported to top_combinations_for_validation.json")
print("Run Cell 12 to export PDB files for downstream validation.")


In [ ]:
with open(DESIGNS_DIR / 'top_combinations_for_validation.json') as f:
    top_combinations = json.load(f)

validation_dir = DESIGNS_DIR / 'validation_inputs'
validation_dir.mkdir(parents=True, exist_ok=True)

import shutil

export_rows = []
for entry in top_combinations:
    rank = entry['rank']
    combination = entry['combination']
    best_pae = entry['best_pae']
    output_dir = Path(entry['output_dir']) if entry.get('output_dir') else None

    pdb_file = None
    if output_dir is not None and output_dir.exists():
        pdb_candidates = sorted(output_dir.rglob('*.pdb'))
        if pdb_candidates:
            pdb_file = pdb_candidates[0]

    if pdb_file is not None:
        dest = validation_dir / f'top_{rank:02d}.pdb'
        shutil.copy2(pdb_file, dest)
        export_rows.append({'rank': rank, 'combination': combination, 'best_pae': best_pae,
                             'pdb_file': str(dest)})
    else:
        export_rows.append({'rank': rank, 'combination': combination, 'best_pae': best_pae,
                             'pdb_file': 'NOT FOUND'})

export_df = pd.DataFrame(export_rows)
display(export_df)

print("Validation inputs ready. Proceed to Notebook 04 (Boltz-2 validation).")
